# Export Functionality Test Notebook

This notebook tests all export-related functionality in `simulation.py` and `simulation_data.py`.

**Test Coverage:**
- DataFrame helper methods (`get_metadata_df`, `get_summary_stats_df`, `get_cultivation_summary_df`, `get_batch_df`)
- `SimulationData.save()` — Excel format with various flag combinations
- `SimulationData.save()` — CSV format
- `SimulationData.save()` — `include_pickle=False`, `add_timestamp=False`
- `SimulationData.save_pickle()` — standalone pickle save
- `SimulationData.load_pickle()` — load and integrity verification
- `CellCultureSimulation.save()` — simulation-level save method
- `CellCultureSimulation.save_pickle()` — simulation-level pickle save
- `get_script_export_dir()` — script-relative export path helper
- Overwrite protection and overwrite flag behaviour
- File existence checks after export
- Cleanup of test files

## 1. Setup and Imports

In [1]:
import sys
import shutil
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation_data import SimulationData, EXPORT_DIR, get_script_export_dir
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")
print(f"  Default EXPORT_DIR: {EXPORT_DIR}")

✓ Imports successful
  Default EXPORT_DIR: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations


## 2. Run a Short Simulation (used across all tests)

In [2]:
# Load parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI"])

# Use BATCH mode — fast single-phase run, no extra strategy params needed
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.BATCH,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False  # Full data for export completeness
)

print(f"✓ Simulation initialised  |  mode={sim.operating_mode}  |  kinetics={sim.kinetics}")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).
✓ Simulation initialised  |  mode=Batch  |  kinetics=Stoichiometric


In [3]:
# Run 2 batches — enough to exercise multi-batch export paths
results = sim.run_simulation(number_of_batches=2)

print(f"✓ Simulation complete")
print(f"  Batches: {len(sim.data.batches)}")
print(f"  STY:     {sim.data.summary_stats['STY_kg_m3_day']:.4f} kg/m³/day")

run_simulation execution time: 119.986 ms
✓ Simulation complete
  Batches: 2
  STY:     0.3666 kg/m³/day


## 3. DataFrame Helper Methods

Tests `get_metadata_df`, `get_summary_stats_df`, `get_cultivation_summary_df`, `get_batch_df`.

In [4]:
# --- get_metadata_df ---
meta_df = sim.data.get_metadata_df(verbose=True)

assert meta_df is not None and len(meta_df) > 0,        "metadata df is empty"
assert 'simulation_type' in meta_df.columns,           "missing simulation_type column"
print("✓ get_metadata_df OK")


=== METADATA ===
      simulation_timestamp simulation_type feed_strategy perfusion_strategy
2026-08-13T17:00:28.941860           Batch            NA                 NA
✓ get_metadata_df OK


In [5]:
# --- get_summary_stats_df ---
stats_df = sim.data.get_summary_stats_df(verbose=True)

assert stats_df is not None and len(stats_df) > 0,      "summary stats df is empty"
assert 'Metric' in stats_df.columns,                   "missing Metric column"
assert 'Value' in stats_df.columns,                    "missing Value column"
assert 'Unit'  in stats_df.columns,                    "missing Unit column"
print("✓ get_summary_stats_df OK")


=== SUMMARY STATISTICS ===
                          Metric                                                                                  Description     Value                 Unit
               Harvested Biomass   Productive dry biomass harvested over the full simulation, excluding non-harvested losses.  5.882726                   kg
           Harvested Wet Biomass Productive wet cell mass harvested over the full simulation, excluding non-harvested losses. 39.218175               kg WCM
                             STY             Space-time yield based on harvested dry biomass, reactor size, and elapsed time.  0.366627            kg/m3/day
                         WCM STY           Space-time yield based on harvested wet cell mass, reactor size, and elapsed time.  2.444183        kg WCM/m3/day
    Medium Conversion Efficiency                       Total process medium input per harvested dry biomass; lower is better.  0.660319            m3/kg DCM
WCM Medium Conversion Efficien

In [6]:
# --- get_cultivation_summary_df ---
cult_df = sim.data.get_cultivation_summary_df(verbose=True)

assert cult_df is not None and len(cult_df) == 2,       "expected 2 rows (one per batch)"
assert 'Batch No.' in cult_df.columns,                 "missing Batch No. column"
assert 'Cultivation Time (h)' in cult_df.columns,                  "missing Time (h) column"
print("✓ get_cultivation_summary_df OK")


=== CULTIVATION SUMMARY ===
 Batch No. Termination Reason  Cultivation Time (h)  Batch Start Time (h)  Batch End Time (h)  Accum. Cultivation Time (d)  Final Glucose (mmol/L)  Final Glutamine (mmol/L)  Final Lactate (mmol/L)  Final Ammonia (mmol/L)  Final Operating Volume (L)  Batch Medium Use (L)  Feed Medium Use (L)  Base Addition Use (L)  Acid Addition Use (L)  Total Medium Use (L)  Accum. Medium Use (L)  Total Outlet Volume (L)  Accum. Total Outlet Volume (L)  Volume Closure Residual (L)  Glucose Added (mmol)  Accum. Glucose Added (mmol)  Glutamine Added (mmol)  Accum. Glutamine Added (mmol)  AA Added (mmol)  Accum. AA Added (mmol)  Total Inlet Use (L)  Accum. Base Addition Use (L)  Accum. Acid Addition Use (L)  Accum. Total Inlet Use (L)  Harvest Volume (L)  Accum. Harvest Volume (L)  Final Cell Density (10^6 cells/mL)  Final Cell Density (g/L)  Cells Harvested (10^12 Cells) Cells Harvested (kg)  Accum. Cell Harvest (10^12 Cells) Accum. Biomass Harvest (kg)
         1       Aa de

In [7]:
# --- get_batch_df ---
batch_df = sim.data.get_batch_df(batch_number=1, verbose=True)

assert batch_df is not None and len(batch_df) > 0,     "batch df is empty"
assert 'Time (h)' in batch_df.columns,                 "missing Time (h) column"
assert 'Cell Density (10^6 cells/mL)' in batch_df.columns, "missing cell density column"

# Test invalid batch number returns None gracefully
missing = sim.data.get_batch(batch_number=99)
assert missing is None, "get_batch with invalid number should return None"
print("✓ get_batch_df OK")
print("✓ get_batch(invalid) returns None OK")


=== BATCH 1 DATA ===
 Time (h)  Cell Density (10^6 cells/mL)  Glucose (mmol/L)  Glutamine (mmol/L)  Lactate (mmol/L)  Ammonia (mmol/L)  Volume (L)  Dilution Rate (1/h)  Base Dilution Rate (1/h)  Total Inlet Dilution Rate (1/h)  Acid Dilution Rate (1/h)  Feed Medium Use (L)  Base Addition Use (L)  Acid Addition Use (L)  Total Outlet Volume (L)  Permeate Volume (L)  Glucose Out (mmol)  Glutamine Out (mmol)  AA Out (mmol)  Glucose Net (mmol)  Glutamine Net (mmol)  AA Net (mmol)  Lactate Out (mmol)  Ammonium Out (mmol)  O2 In (mmol)  O2 Out (mmol)  CT In (mmol)  CT Out (mmol)  Lactate Net (mmol)  Ammonium Net (mmol)  O2 Net (mmol)  CT Net (mmol)  Water Produced (mmol)  Heat Generated (kJ)  Power/Volume (W/m^3)  Superficial Gas Velocity (m/s)  Y_Glc/X (mol/mol)  Y_Lac/Glc (mol/mol)  Y_Gln/Glc (mol/mol)  Y_O2/Glc (mol/mol)  Y_CO2/Glc (mol/mol)  Y_AA/Glc (mol/mol)  Y_Amm/Gln (mol/mol)  q_H2O (mmol/cell/h)  Heat Rate (kJ/L/h)
 0.000000                      0.600000         25.000000          

## 4. `SimulationData.save()` — Excel Format

Tests the `excel` output path with various flag combinations.

In [8]:
# --- 4a: Full Excel export with timestamp (default behaviour) ---
xlsx_path = sim.data.save(
    filename="_test_export_excel_full",
    file_format="excel",
    add_timestamp=True,
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=True,
    include_pickle=True,    # Also saves a .pkl alongside
    overwrite=True
)

assert xlsx_path.exists(),                          f"Excel file not found at {xlsx_path}"
assert xlsx_path.suffix == '.xlsx',                 "exported file should have .xlsx extension"

# Parameters YAML should be alongside
params_path = xlsx_path.with_suffix('').with_suffix('.parameters.yaml')
assert params_path.exists(),                        f"parameters YAML not found at {params_path}"

# Pickle file should be alongside
pkl_path = xlsx_path.with_suffix('').with_suffix('.pkl')
assert pkl_path.exists(),                           f"pickle file not found at {pkl_path}"

print(f"✓ Full Excel export OK  →  {xlsx_path.name}")
print(f"✓ Parameters YAML saved  →  {params_path.name}")
print(f"✓ Pickle saved alongside →  {pkl_path.name}")

Simulation data exported to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1700\_test_export_excel_full_20260813_1700.xlsx
Parameters saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1700\_test_export_excel_full_20260813_1700.parameters.yaml
SimulationData object saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1700\_test_export_excel_full_20260813_1700.pkl
✓ Full Excel export OK  →  _test_export_excel_full_20260813_1700.xlsx
✓ Parameters YAML saved  →  _test_export_excel_full_20260813_1700.parameters.yaml
✓ Pickle saved alongside →  _test_export_excel_full_20260813_1700.pkl


In [9]:
# --- 4b: Excel export without timestamp, no pickle, no parameters ---
xlsx_minimal_path = sim.data.save(
    filename="_test_export_excel_minimal",
    file_format="excel",
    add_timestamp=True,
    include_metadata=True,
    include_summary_stats=True,
    include_mass_balance=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=True,
    include_pickle=False,
    overwrite=True
)

assert xlsx_minimal_path.exists(),                  f"Minimal Excel file not found at {xlsx_minimal_path}"

# No pickle should exist alongside
pkl_should_not_exist = xlsx_minimal_path.with_suffix('').with_suffix('.pkl')
assert not pkl_should_not_exist.exists(),           "Pickle should NOT exist when include_pickle=False"

print(f"✓ Minimal Excel export OK  →  {xlsx_minimal_path.name}")
print(f"✓ No spurious pickle file  →  confirmed")

Simulation data exported to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_minimal_20260813_1700\_test_export_excel_minimal_20260813_1700.xlsx
Parameters saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_minimal_20260813_1700\_test_export_excel_minimal_20260813_1700.parameters.yaml
✓ Minimal Excel export OK  →  _test_export_excel_minimal_20260813_1700.xlsx
✓ No spurious pickle file  →  confirmed


In [10]:
# --- 4c: Overwrite protection test ---
import traceback

try:
    sim.data.save(
        filename="_test_export_excel_minimal",
        file_format="excel",
        add_timestamp=False,
        include_pickle=False,
        overwrite=False   # Should raise since file already exists
    )
    print("✗ Expected an error for overwrite=False on existing file — NONE raised")
except (FileExistsError, Exception) as exc:
    print(f"✓ Overwrite protection raised: {type(exc).__name__}: {exc}")

✓ Overwrite protection raised: FileExistsError: File c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_minimal\_test_export_excel_minimal.xlsx already exists. Set overwrite=True to replace it.


## 5. `SimulationData.save()` — CSV Format

In [11]:
# --- 5a: Full CSV export ---
csv_path = sim.data.save(
    filename="_test_export_csv_full",
    file_format="csv",
    add_timestamp=False,
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=True,
    include_pickle=False,
    overwrite=True
)

# csv_path points to the export directory folder (or a manifest file); check that CSV files were created
csv_dir = csv_path.parent if csv_path.suffix == '.csv' else csv_path
csv_files = list(csv_dir.rglob("*.csv"))
assert len(csv_files) > 0,                           f"No CSV files found under {csv_dir}"
print(f"✓ CSV export OK  →  {len(csv_files)} .csv file(s) in {csv_dir.name}/")
for f in csv_files:
    print(f"    {f.name}")

Simulation data exported to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_csv_full (7 files)
Parameters saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_csv_full.parameters.yaml
✓ CSV export OK  →  8 .csv file(s) in _test_export_csv_full/
    01_metadata.csv
    02_summary_statistics.csv
    03_cultivation_summary.csv
    03_projected_totals.csv
    04_simulated_window_totals.csv
    05_cultivation_summary.csv
    batch_01.csv
    batch_02.csv


## 6. `SimulationData.save_pickle()` — Standalone Pickle Save

In [12]:
# --- 6a: Save pickle to default EXPORT_DIR ---
pkl_standalone = sim.data.save_pickle("_test_export_standalone_pkl", overwrite=True)

assert pkl_standalone.exists(),                      f"Standalone pickle not found at {pkl_standalone}"
assert pkl_standalone.suffix == '.pkl',              "pickle file should have .pkl extension"
print(f"✓ save_pickle OK  →  {pkl_standalone}")
print(f"  File size: {pkl_standalone.stat().st_size / 1024:.1f} KB")

SimulationData object saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_standalone_pkl.pkl
✓ save_pickle OK  →  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_standalone_pkl.pkl
  File size: 199.9 KB


In [13]:
# --- 6b: Overwrite protection on save_pickle ---
try:
    sim.data.save_pickle("_test_export_standalone_pkl", overwrite=False)
    print("✗ Expected error for overwrite=False — NONE raised")
except (FileExistsError, Exception) as exc:
    print(f"✓ save_pickle overwrite protection raised: {type(exc).__name__}")

✓ save_pickle overwrite protection raised: FileExistsError


In [14]:
# --- 6c: Save pickle to custom directory (script-relative) ---
custom_dir = Path.cwd() / "_test_custom_export"
pkl_custom = sim.data.save_pickle("_test_export_custom_pkl", overwrite=True, base_export_dir=custom_dir)

assert pkl_custom.exists(),                          f"Custom dir pickle not found at {pkl_custom}"
assert pkl_custom.parent == custom_dir,              f"Expected parent {custom_dir}, got {pkl_custom.parent}"
print(f"✓ save_pickle to custom dir OK  →  {pkl_custom}")

SimulationData object saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\_test_custom_export\_test_export_custom_pkl.pkl
✓ save_pickle to custom dir OK  →  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\_test_custom_export\_test_export_custom_pkl.pkl


## 7. `SimulationData.load_pickle()` — Load and Data Integrity

In [15]:
# --- 7a: Load by filename (searches EXPORT_DIR recursively) ---
loaded = SimulationData.load_pickle("_test_export_standalone_pkl")

assert loaded is not None,                           "load_pickle returned None"
assert isinstance(loaded, SimulationData),           "loaded object is not a SimulationData instance"
print(f"✓ load_pickle OK  →  type={type(loaded).__name__}")

SimulationData object loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_standalone_pkl.pkl
  - Simulation type: Batch
  - Number of batches: 2
  - Timestamp: 2026-08-13 17:00:28
✓ load_pickle OK  →  type=SimulationData


In [16]:
# --- 7b: Data integrity checks ---
assert len(loaded.batches) == len(sim.data.batches),     "batch count mismatch after load"
assert loaded.simulation_type == sim.data.simulation_type, "simulation_type mismatch after load"
assert loaded.parameters is not None,                    "parameters not preserved in pickle"

# Check summary stats round-trip
for key in sim.data.summary_stats:
    orig_val  = sim.data.summary_stats[key]
    loaded_val = loaded.summary_stats[key]
    if isinstance(orig_val, float):
        assert abs(orig_val - loaded_val) < 1e-12, f"summary_stats[{key}] mismatch: {orig_val} vs {loaded_val}"
    else:
        assert orig_val == loaded_val, f"summary_stats[{key}] mismatch: {orig_val} vs {loaded_val}"

print("✓ Summary stats round-trip OK")

# Check numpy array shape and values for batch 1
orig_batch   = sim.data.batches[0]
loaded_batch = loaded.batches[0]

assert orig_batch.y_points.shape == loaded_batch.y_points.shape, "y_points shape mismatch"
assert np.allclose(orig_batch.t_points, loaded_batch.t_points),  "t_points values mismatch"
assert np.allclose(orig_batch.y_points, loaded_batch.y_points),  "y_points values mismatch"
assert orig_batch.Xv_final == loaded_batch.Xv_final,              "Xv_final mismatch"
assert orig_batch.V_final  == loaded_batch.V_final,               "V_final mismatch"

print("✓ Batch state arrays round-trip OK")

# Optional arrays (e.g. mu_list) should also be preserved
if orig_batch.mu_list is not None:
    assert loaded_batch.mu_list is not None,           "mu_list lost in pickling"
    assert np.allclose(orig_batch.mu_list, loaded_batch.mu_list), "mu_list values mismatch"
    print("✓ Kinetic rate arrays (mu_list) round-trip OK")

print(f"\nAll data integrity checks passed for {len(loaded.batches)} loaded batch(es).")

✓ Summary stats round-trip OK
✓ Batch state arrays round-trip OK
✓ Kinetic rate arrays (mu_list) round-trip OK

All data integrity checks passed for 2 loaded batch(es).


In [17]:
# --- 7c: Load by full path ---
loaded_by_path = SimulationData.load_pickle(pkl_standalone)
assert isinstance(loaded_by_path, SimulationData),   "load_pickle from Path returned wrong type"
print("✓ load_pickle by full Path OK")

SimulationData object loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_standalone_pkl.pkl
  - Simulation type: Batch
  - Number of batches: 2
  - Timestamp: 2026-08-13 17:00:28
✓ load_pickle by full Path OK


In [18]:
# --- 7d: All SimulationData API methods work on loaded object ---
meta_df_loaded  = loaded.get_metadata_df()
stats_df_loaded = loaded.get_summary_stats_df()
cult_df_loaded  = loaded.get_cultivation_summary_df()
batch_df_loaded = loaded.get_batch_df(batch_number=1)

assert meta_df_loaded  is not None and len(meta_df_loaded)  > 0, "metadata df empty on loaded data"
assert stats_df_loaded is not None and len(stats_df_loaded) > 0, "summary stats df empty on loaded data"
assert cult_df_loaded  is not None and len(cult_df_loaded)  > 0, "cultivation df empty on loaded data"
assert batch_df_loaded is not None and len(batch_df_loaded) > 0, "batch df empty on loaded data"

print("✓ All DataFrame methods work on loaded SimulationData")

✓ All DataFrame methods work on loaded SimulationData


## 8. `CellCultureSimulation.save()` — Simulation-Level Save

In [19]:
# --- 8a: sim.save() forwards to SimulationData.save() ---
sim_save_path = sim.save(
    filename="_test_sim_save_excel",
    file_format="excel",
    add_timestamp=True,
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=True,
    include_pickle=True,
    include_opt_parameters=False,
    overwrite=True
)

assert sim_save_path.exists(),                       f"sim.save() file not found at {sim_save_path}"
print(f"✓ sim.save() OK  →  {sim_save_path.name}")

Simulation data exported to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_excel_20260813_1700\opt_20260813_1700.xlsx
Parameters saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_excel_20260813_1700\opt_20260813_1700.parameters.yaml
CellCultureSimulation pickle saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_excel_20260813_1700\opt_20260813_1700.pkl
✓ sim.save() OK  →  opt_20260813_1700.xlsx


## 9. `CellCultureSimulation.save_pickle()` — Simulation-Level Pickle Save

In [20]:
# --- 9a: sim.save_pickle() delegates to SimulationData.save_pickle() ---
sim_pkl_path = sim.save_pickle("_test_sim_save_pkl", overwrite=True)

assert sim_pkl_path.exists(),                        f"sim.save_pickle() file not found at {sim_pkl_path}"
assert sim_pkl_path.suffix == '.pkl',               "sim.save_pickle() should produce a .pkl file"
print(f"✓ sim.save_pickle() OK  →  {sim_pkl_path}")

# Verify load works on sim-level pickle
reloaded = CellCultureSimulation.load_pickle("_test_sim_save_pkl")
assert isinstance(reloaded, CellCultureSimulation), "Reloaded sim pickle is not a SimulationData instance"
print("✓ Reload of sim.save_pickle() file OK")

CellCultureSimulation saved to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_pkl.pkl
✓ sim.save_pickle() OK  →  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_pkl.pkl
CellCultureSimulation loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_sim_save_pkl.pkl
✓ Reload of sim.save_pickle() file OK


## 10. `get_script_export_dir()` Helper

In [21]:
# --- 10a: Default subdir ---
script_dir = get_script_export_dir()
print(f"get_script_export_dir()          →  {script_dir}")
assert isinstance(script_dir, Path),                 "should return a Path object"
assert script_dir.name == 'simulation',             f"default subdir should be 'simulation', got '{script_dir.name}'"
print("✓ get_script_export_dir() default subdir OK")

# --- 10b: Custom subdir ---
custom = get_script_export_dir(subdir='my_exports')
assert custom.name == 'my_exports',                  f"custom subdir not applied, got '{custom.name}'"
print("✓ get_script_export_dir(subdir='my_exports') OK")

# --- 10c: Use get_script_export_dir() as export_dir in save ---
export_to_script_dir = sim.data.save(
    filename="_test_export_script_dir",
    file_format="excel",
    add_timestamp=False,
    include_pickle=False,
    include_parameters=False,
    overwrite=True,
    export_dir=get_script_export_dir(subdir='_test_script_exports')
)

assert export_to_script_dir.exists(),                f"script-dir export not found at {export_to_script_dir}"
print(f"✓ save() with get_script_export_dir() OK  →  {export_to_script_dir}")

get_script_export_dir()          →  C:\Users\btumulero\AppData\Local\Temp\ipykernel_18776\simulation
✓ get_script_export_dir() default subdir OK
✓ get_script_export_dir(subdir='my_exports') OK
Simulation data exported to: C:\Users\btumulero\AppData\Local\Temp\ipykernel_18776\_test_script_exports\_test_export_script_dir\_test_export_script_dir.xlsx
✓ save() with get_script_export_dir() OK  →  C:\Users\btumulero\AppData\Local\Temp\ipykernel_18776\_test_script_exports\_test_export_script_dir\_test_export_script_dir.xlsx


## 11. Export to Custom `export_dir` Path

In [22]:
# --- 11a: save() to a custom export_dir ---
custom_export = Path.cwd() / "_test_custom_export"

custom_xlsx = sim.data.save(
    filename="_test_export_custom_dir_excel",
    file_format="excel",
    add_timestamp=False,
    include_pickle=False,
    include_parameters=False,
    overwrite=True,
    export_dir=custom_export
)

assert custom_xlsx.exists(),                         f"Custom-dir Excel not found at {custom_xlsx}"
assert str(custom_export) in str(custom_xlsx),       "exported file not inside custom_export dir"
print(f"✓ Custom export_dir save OK  →  {custom_xlsx}")

Simulation data exported to: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\_test_custom_export\_test_export_custom_dir_excel\_test_export_custom_dir_excel.xlsx
✓ Custom export_dir save OK  →  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\test\_test_custom_export\_test_export_custom_dir_excel\_test_export_custom_dir_excel.xlsx


## 12. Summary

In [23]:
print("=" * 60)
print(" EXPORT FUNCTIONALITY TEST SUMMARY")
print("=" * 60)
print(" PASSED:")
print("   ✓  get_metadata_df()")
print("   ✓  get_summary_stats_df()")
print("   ✓  get_cultivation_summary_df()")
print("   ✓  get_batch_df()")
print("   ✓  SimulationData.save() — Excel (full flags)")
print("   ✓  SimulationData.save() — Excel (minimal flags)")
print("   ✓  SimulationData.save() — Excel overwrite protection")
print("   ✓  SimulationData.save() — CSV format")
print("   ✓  SimulationData.save_pickle() — default dir")
print("   ✓  SimulationData.save_pickle() — custom dir")
print("   ✓  SimulationData.save_pickle() — overwrite protection")
print("   ✓  SimulationData.load_pickle() — by filename")
print("   ✓  SimulationData.load_pickle() — by full Path")
print("   ✓  Data integrity after pickle round-trip")
print("   ✓  API methods on loaded SimulationData")
print("   ✓  CellCultureSimulation.save()")
print("   ✓  CellCultureSimulation.save_pickle()")
print("   ✓  get_script_export_dir()")
print("   ✓  save() with custom export_dir")
print("=" * 60)

 EXPORT FUNCTIONALITY TEST SUMMARY
 PASSED:
   ✓  get_metadata_df()
   ✓  get_summary_stats_df()
   ✓  get_cultivation_summary_df()
   ✓  get_batch_df()
   ✓  SimulationData.save() — Excel (full flags)
   ✓  SimulationData.save() — Excel (minimal flags)
   ✓  SimulationData.save() — Excel overwrite protection
   ✓  SimulationData.save() — CSV format
   ✓  SimulationData.save_pickle() — default dir
   ✓  SimulationData.save_pickle() — custom dir
   ✓  SimulationData.save_pickle() — overwrite protection
   ✓  SimulationData.load_pickle() — by filename
   ✓  SimulationData.load_pickle() — by full Path
   ✓  Data integrity after pickle round-trip
   ✓  API methods on loaded SimulationData
   ✓  CellCultureSimulation.save()
   ✓  CellCultureSimulation.save_pickle()
   ✓  get_script_export_dir()
   ✓  save() with custom export_dir


## 13. Cleanup Test Files

In [24]:
# Remove all test artefacts created during this notebook run.
# Set DRY_RUN = True to preview what would be deleted without deleting.

DRY_RUN = True

to_delete = [
    # Excel / CSV export subdirs in default EXPORT_DIR
    *EXPORT_DIR.glob("_test_export_excel_full*"),
    *EXPORT_DIR.glob("_test_export_excel_minimal*"),
    *EXPORT_DIR.glob("_test_export_csv_full*"),
    *EXPORT_DIR.glob("_test_sim_save_excel*"),
    # Standalone pickles in default EXPORT_DIR
    EXPORT_DIR / "_test_export_standalone_pkl.pkl",
    EXPORT_DIR / "_test_sim_save_pkl.pkl",
    # Pickle saved alongside Excel export (nested inside timestamped subdir)
    *EXPORT_DIR.glob("_test_export_excel_full*/**/*.pkl"),
    *EXPORT_DIR.glob("_test_sim_save_excel*/**/*.pkl"),
    # Custom directory created during the test
    Path.cwd() / "_test_custom_export",
    # Script-relative export dir
    Path.cwd() / "_test_script_exports",
]

deleted, skipped = [], []
for path in to_delete:
    if path.exists():
        if not DRY_RUN:
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
        deleted.append(path)
    else:
        skipped.append(path)

action = "Would delete" if DRY_RUN else "Deleted"
print(f"{action} {len(deleted)} item(s):")
for p in deleted:
    print(f"  {p}")
if skipped:
    print(f"\nSkipped {len(skipped)} item(s) (not found):")
    for p in skipped:
        print(f"  {p}")

print("\n✓ Cleanup complete" if not DRY_RUN else "\n(Dry run — nothing deleted)")

Would delete 31 item(s):
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260609_1519
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260611_1042
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260624_1125
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1448
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1450
  c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\_test_export_excel_full_20260813_1451
  c:\Users\btumulero\OneDrive - Delft University of Techn